<a href="https://colab.research.google.com/github/ivanduzunov/AI-Agents-and-Workflows-for-Developers/blob/main/final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# festival planner
!pip install q langchain langchain-openai

In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.types import Send, interrupt, Command
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, Literal, List, Annotated, Dict, Optional
from IPython.display import Image
from google.colab import userdata

openai_key = userdata.get('OPENAI_KEY')

In [ ]:
# research node - find festival
#

In [ ]:
"""
START
  │
  ▼
Research
  │
  ▼
Select Festival ── interrupt()
  │
  ▼
Find Accommodation
  │
  ▼
Select Accommodation ── interrupt()
  │
  ▼
Research Travel
  │
  ▼
Select Flight ── interrupt()
  │
  ▼
Confirm Purchase ── interrupt()
  │
  ▼
Buy Tickets
  │
  ▼
END
"""

In [7]:
class ConcertPlannerState(TypedDict):
    user_request: str

    festivals: list
    selected_festival: Optional[dict]

    accommodation_type: Optional[str]
    accommodations: list
    selected_accommodation: Optional[dict]

    flights: list
    selected_flight: Optional[dict]

    purchase_approved: bool

    final_result: Optional[str]

In [11]:
def research(state: ConcertPlannerState):
    user_request = state["user_request"]

    # AI agent + web search tool
    festivals = festival_agent.invoke({
        "request": user_request
    })

    return {
        "festivals": festivals
    }

def select_festival(state: ConcertPlannerState):

    festivals = state["festivals"]

    user_selection = interrupt({
        "type": "festival_selection",
        "message": "Choose a festival:",
        "options": festivals
    })

    selected_festival = festivals[user_selection]

    return {
        "selected_festival": selected_festival
    }

def find_accommodation(state: ConcertPlannerState):

    festival = state["selected_festival"]

    accommodation_type = state["accommodation_type"]

    accommodations = accommodation_agent.invoke({
        "festival": festival,
        "type": accommodation_type
    })

    return {
        "accommodations": accommodations
    }

def select_accommodation_type(state: ConcertPlannerState):

    accommodation_type = interrupt({
        "type": "accommodation_type",
        "message": "Where would you like to stay?",
        "options": [
            "hotel",
            "camping"
        ]
    })

    return {
        "accommodation_type": accommodation_type
    }

def select_accommodation(state: ConcertPlannerState):

    accommodations = state["accommodations"]

    user_selection = interrupt({
        "type": "accommodation_selection",
        "message": "Choose your accommodation:",
        "options": accommodations
    })

    selected_accommodation = accommodations[user_selection]

    return {
        "selected_accommodation": selected_accommodation
    }

def research_travel(state: ConcertPlannerState):

    festival = state["selected_festival"]
    accommodation = state["selected_accommodation"]

    flights = travel_agent.invoke({
        "festival": festival,
        "accommodation": accommodation
    })

    return {
        "flights": flights
    }

def select_flight(state: ConcertPlannerState):

    flights = state["flights"]

    user_selection = interrupt({
        "type": "flight_selection",
        "message": "Choose your flight:",
        "options": flights
    })

    selected_flight = flights[user_selection]

    return {
        "selected_flight": selected_flight
    }

def confirm_purchase(state: ConcertPlannerState):

    festival = state["selected_festival"]
    accommodation = state["selected_accommodation"]
    flight = state["selected_flight"]

    total_price = (
        festival["price"]
        + accommodation["price"]
        + flight["price"]
    )

    approval = interrupt({
        "type": "purchase_confirmation",
        "message": "Please review your trip before purchase.",
        "festival": festival,
        "accommodation": accommodation,
        "flight": flight,
        "total_price": total_price
    })

    return {
        "purchase_approved": approval
    }

def buy_tickets(state: ConcertPlannerState):

    if not state["purchase_approved"]:
        return {
            "final_result": "Purchase cancelled by user."
        }

    festival = state["selected_festival"]
    accommodation = state["selected_accommodation"]
    flight = state["selected_flight"]

    result = {
        "festival": festival,
        "accommodation": accommodation,
        "flight": flight
    }

    return {
        "final_result": result
    }

In [ ]:
# graph_builder = StateGraph(ConcertPlannerState)
# graph_builder.add_node("Research", empty_fn)
# graph_builder.add_node("Find accomodation", empty_fn)
# graph_builder.add_node("Research travel", empty_fn)
# graph_builder.add_node("Buy tickets", empty_fn)

# graph_builder.add_edge(START, "Research")
# graph_builder.add_edge("Research", "Find accomodation")
# graph_builder.add_edge("Find accomodation", "Research travel")
# graph_builder.add_edge("Research travel", "Buy tickets")


# graph = graph_builder.compile(debug=True)

graph_builder = StateGraph(ConcertPlannerState)
graph_builder.add_node("Research", research)
graph_builder.add_node("Select Festival", select_festival)
graph_builder.add_node("Select Accommodation Type", select_accommodation_type)
graph_builder.add_node("Find Accommodation", find_accommodation)
graph_builder.add_node("Select Accommodation", select_accommodation)
graph_builder.add_node("Research Travel", research_travel)
graph_builder.add_node("Select Flight", select_flight)
graph_builder.add_node("Confirm Purchase", confirm_purchase)
graph_builder.add_node("Buy Tickets", buy_tickets)

graph_builder.add_edge(START, "Research")
graph_builder.add_edge("Research", "Select Festival")
graph_builder.add_edge("Select Festival", "Select Accommodation Type")
graph_builder.add_edge("Select Accommodation Type", "Find Accommodation")
graph_builder.add_edge("Find Accommodation", "Select Accommodation")
graph_builder.add_edge("Select Accommodation", "Research Travel")
graph_builder.add_edge("Research Travel", "Select Flight")
graph_builder.add_edge("Select Flight", "Confirm Purchase")
graph_builder.add_edge("Confirm Purchase", "Buy Tickets")
graph_builder.add_edge("Buy Tickets", END)